In [ ]:
# ============================================================================
# 3 - Normalize CR (Course Reviews) to Silver
# ----------------------------------------------------------------------------
# Reads the raw student course reviews that land in ADLS Gen2 (surfaced through
# the BronzeLakehouse "data" shortcut), calls the Azure AI Foundry chat model
# deployed by azure-scripts (main.bicep -> modelDeployments -> "gpt-5") to
# ANONYMIZE and rewrite each free-text review so an individual student can no
# longer be identified or traced, drops the direct identifiers
# (studentId / studentName), and writes a de-identified Delta table to the
# SilverLakehouse.
#
# Per-environment configuration (Foundry endpoint, model deployment, API version,
# and optional Key Vault coordinates for the API key) is supplied by the
# "StudentDemoVariables" Fabric Variable Library. The active value set
# is chosen per workspace, so different deployments use
# different endpoints and keys with no code change.
# ============================================================================

In [ ]:
# Fabric Variable Library that supplies the per-environment configuration
# (Foundry endpoint, model deployment, API version, and optional Key Vault
# coordinates for the API key). The ACTIVE value set of this library is selected
# per workspace / deployment-pipeline stage, so the same
# notebook targets different endpoints and keys without any code change.
# See fabric-workspace/StudentDemoVariables.VariableLibrary.
variable_library_name = "StudentDemoVariables"

# Optional run-time overrides. Leave blank to use the value from the active value
# set of the variable library above. These let you point an ad-hoc run at a
# different endpoint/key without editing the library.
foundry_endpoint_override = ""
foundry_api_key_override = ""

# Source lakehouse that exposes the ADLS course-review files via its "data"
# shortcut (/Files/data). Resolved by name at run time (no GUID committed).
source_lakehouse = "BronzeLakehouse"

# Path to the course reviews, relative to the source lakehouse Files area.
source_files_subpath = "data/course_reviews.parquet"

# Target Silver table (schema-enabled lakehouse; "dbo" is the default schema).
target_table = "course_reviews"

# Cap the number of reviews processed in one run (0 = all).
max_reviews = 0

# Number of parallel model calls.
max_workers = 4

In [ ]:
import json
import uuid
import requests
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
)

# --- Resolve environment-specific config from the Fabric Variable Library ----
# The active value set (DEV / TEST / PROD / ...) is chosen per workspace, so this
# same code runs against different endpoints and keys per deployment.
vl = notebookutils.variableLibrary.getLibrary(variable_library_name)

foundry_endpoint = (foundry_endpoint_override or vl.getVariable("foundry_endpoint") or "").rstrip("/")
chat_deployment = vl.getVariable("chat_deployment")
api_version = vl.getVariable("api_version")
key_vault_uri = (vl.getVariable("key_vault_uri") or "").strip()
foundry_key_secret_name = (vl.getVariable("foundry_key_secret_name") or "").strip()

if not foundry_endpoint:
    raise ValueError(
        f"foundry_endpoint is empty. Set it in the active value set of the "
        f"'{variable_library_name}' variable library (or pass "
        f"foundry_endpoint_override)."
    )

# Resolve the API key: explicit override > Key Vault secret > Entra token (blank).
# The key value itself is never committed - only the Key Vault coordinates live
# in the variable library.
if foundry_api_key_override:
    foundry_api_key = foundry_api_key_override
elif key_vault_uri and foundry_key_secret_name:
    foundry_api_key = notebookutils.credentials.getSecret(
        key_vault_uri, foundry_key_secret_name
    )
else:
    foundry_api_key = ""

print(f"Endpoint       : {foundry_endpoint}")
print(f"Deployment     : {chat_deployment}")
print(f"API version    : {api_version}")
print(f"Auth mode      : {'api-key' if foundry_api_key else 'Entra token'}")

In [ ]:
# --- Read the raw course reviews from the Bronze "data" shortcut -------------
src = notebookutils.lakehouse.get(source_lakehouse)
props = src.get("properties", {})
files_root = props.get("oneLakeFilesPath") or (
    props.get("abfsPath", "").rstrip("/") + "/Files"
)
source_uri = f"{files_root.rstrip('/')}/{source_files_subpath.lstrip('/')}"
print(f"Reading course reviews from: {source_uri}")

if source_files_subpath.lower().endswith(".parquet"):
    raw_df = spark.read.parquet(source_uri)
else:
    raw_df = spark.read.option("multiline", "true").json(source_uri)

total = raw_df.count()
print(f"Loaded {total} course reviews.")
raw_df.printSchema()

In [ ]:
# --- Azure AI Foundry chat client + anonymization prompt ---------------------
CHAT_URL = (
    f"{foundry_endpoint}/openai/deployments/{chat_deployment}"
    f"/chat/completions?api-version={api_version}"
)


def _auth_headers():
    """Prefer an explicit API key; otherwise use the caller's Entra token."""
    if foundry_api_key:
        return {"api-key": foundry_api_key, "Content-Type": "application/json"}
    token = notebookutils.credentials.getToken("https://cognitiveservices.azure.com")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


SYSTEM_PROMPT = (
    "You are a privacy-preserving data steward for a university. You are given a "
    "single student's free-text course-review response. Correct the grammar and "
    "rewrite it so that the individual student can NOT be identified or traced, "
    "while preserving the overall sentiment, the perceived course difficulty, and "
    "the substantive feedback about the course. Rules: fix spelling and grammar; "
    "remove or generalize any personal names, gendered pronouns, unique personal "
    "circumstances, specific dates, locations, and any other identifying detail. "
    "Do not invent new facts and do not add a preamble. Return ONLY the rewritten, "
    "anonymized review text."
)


def anonymize_text(text):
    if text is None or str(text).strip() == "":
        return ""
    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(text)},
        ],
        # gpt-5 uses max_completion_tokens and only supports the default temperature.
        "max_completion_tokens": 500,
    }
    resp = requests.post(
        CHAT_URL, headers=_auth_headers(), data=json.dumps(body), timeout=120
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()

In [ ]:
# --- Anonymize every review; drop direct identifiers ------------------------
rows = raw_df.collect()
if max_reviews and max_reviews > 0:
    rows = rows[:max_reviews]
print(f"Anonymizing {len(rows)} reviews via '{chat_deployment}'...")


def _process(row):
    d = row.asDict()
    # studentId and studentName are intentionally NOT carried forward so the
    # student cannot be traced. A fresh surrogate id replaces them.
    return {
        "reviewId": str(uuid.uuid4()),
        "courseId": int(d["courseId"]) if d.get("courseId") is not None else None,
        "courseNumber": d.get("courseNumber"),
        "courseName": d.get("courseName"),
        "letterGrade": d.get("letterGrade"),
        "qualityPoints": float(d["qualityPoints"])
        if d.get("qualityPoints") is not None
        else None,
        "courseDifficulty": d.get("courseDifficulty"),
        "difficultyDescriptor": d.get("difficultyDescriptor"),
        "targetGraduateProgram": d.get("targetGraduateProgram"),
        "reviewDate": str(d["reviewDate"]) if d.get("reviewDate") is not None else None,
        "anonymizedReview": anonymize_text(d.get("studentSurveyResponse")),
    }


with ThreadPoolExecutor(max_workers=max_workers) as pool:
    records = list(pool.map(_process, rows))

print(f"Anonymized {len(records)} reviews.")

In [ ]:
# --- Write the de-identified reviews to the Silver lakehouse ----------------
schema = StructType(
    [
        StructField("reviewId", StringType()),
        StructField("courseId", LongType()),
        StructField("courseNumber", StringType()),
        StructField("courseName", StringType()),
        StructField("letterGrade", StringType()),
        StructField("qualityPoints", DoubleType()),
        StructField("courseDifficulty", StringType()),
        StructField("difficultyDescriptor", StringType()),
        StructField("targetGraduateProgram", StringType()),
        StructField("reviewDate", StringType()),
        StructField("anonymizedReview", StringType()),
    ]
)

out_df = spark.createDataFrame(records, schema=schema)
(
    out_df.write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)
print(f"Wrote {out_df.count()} anonymized reviews to Silver table '{target_table}'.")

In [ ]:
display(spark.table(target_table).limit(10))